# LeetCode #1301: Number of Paths with Max Score

https://leetcode.com/problems/number-of-paths-with-max-score/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(3^{mn})$ | $O(mn)$ |
| **Optimal: 2D DP ★** | $O(mn)$ | $O(mn)$ |

---

## Understanding the Methods

### Brute Force
DFS from `'E'` to `'S'` trying all paths (up, left, diagonal). Exponential branching makes this infeasible for grids larger than ~4×4.

### Optimal: 2D DP ★
Process the grid bottom-right to top-left. `dp[r][c]` stores `(maxScore, pathCount)`. Transition: consider the three predecessors (right, down, diagonal); take the maximum score among reachable predecessors and sum their path counts. Numeric digits contribute to score; `'X'` cells are walls that break paths. Answer is `(dp[0][0].score, dp[0][0].count % MOD)`.

**Constraints:**
* `2 <= board.length == board[0].length <= 100`
* `board[i][j]` is `'E'`, `'S'`, `'X'`, or a digit `'0'`–`'9'`.

## Solutions

### C#

In [ ]:
public class Solution {
    private const int MOD = 1_000_000_007;

    public int[] PathsWithMaxScore(IList<string> board) {
        int n = board.Count;
        int[,] score = new int[n, n];
        long[,] cnt  = new long[n, n];

        // Bottom-right corner is the destination 'E', score 0, 1 path
        score[n-1, n-1] = 0;
        cnt[n-1, n-1]   = 1;

        // Fill bottom row (can only come from the right)
        for (int c = n-2; c >= 0; c--) {
            if (board[n-1][c] == 'X' || cnt[n-1, c+1] == 0) continue;
            score[n-1, c] = score[n-1, c+1] + (board[n-1][c] == 'S' ? 0 : board[n-1][c] - '0');
            cnt[n-1, c]   = cnt[n-1, c+1];
        }
        // Fill right column (can only come from below)
        for (int r = n-2; r >= 0; r--) {
            if (board[r][n-1] == 'X' || cnt[r+1, n-1] == 0) continue;
            score[r, n-1] = score[r+1, n-1] + (board[r][n-1] == 'S' ? 0 : board[r][n-1] - '0');
            cnt[r, n-1]   = cnt[r+1, n-1];
        }

        // Fill interior from bottom-right to top-left
        for (int r = n-2; r >= 0; r--) {
            for (int c = n-2; c >= 0; c--) {
                if (board[r][c] == 'X') continue;
                int cellVal = (board[r][c] == 'S') ? 0 : (board[r][c] - '0');
                int best = -1; long paths = 0;

                // Merge the three successor cells: right, below, diagonal
                Merge(score[r, c+1], cnt[r, c+1], ref best, ref paths);
                Merge(score[r+1, c], cnt[r+1, c], ref best, ref paths);
                Merge(score[r+1, c+1], cnt[r+1, c+1], ref best, ref paths);

                if (best >= 0) { score[r, c] = best + cellVal; cnt[r, c] = paths % MOD; }
            }
        }

        if (cnt[0, 0] == 0) return new int[]{0, 0};
        return new int[]{score[0, 0], (int)(cnt[0, 0] % MOD)};
    }

    private void Merge(int s, long c, ref int best, ref long paths) {
        if (c == 0) return;
        if (s > best)  { best = s; paths = c; }
        else if (s == best) { paths += c; }
    }
}

### Python

In [ ]:
class Solution:
    def pathsWithMaxScore(self, board: list[str]) -> list[int]:
        MOD = 10**9 + 7
        n = len(board)
        score = [[0]*n for _ in range(n)]
        cnt   = [[0]*n for _ in range(n)]
        cnt[n-1][n-1] = 1  # destination 'E': one path, score 0

        def merge(s_new, c_new, best, paths):
            if c_new == 0: return best, paths
            if s_new > best:  return s_new, c_new
            if s_new == best: return best, paths + c_new
            return best, paths

        # Bottom row: right-to-left, only right neighbour accessible
        for c in range(n-2, -1, -1):
            if board[n-1][c] == 'X' or cnt[n-1][c+1] == 0: continue
            val = 0 if board[n-1][c] == 'S' else int(board[n-1][c])
            score[n-1][c] = score[n-1][c+1] + val
            cnt[n-1][c]   = cnt[n-1][c+1]

        # Right column: top-to-bottom, only below neighbour accessible
        for r in range(n-2, -1, -1):
            if board[r][n-1] == 'X' or cnt[r+1][n-1] == 0: continue
            val = 0 if board[r][n-1] == 'S' else int(board[r][n-1])
            score[r][n-1] = score[r+1][n-1] + val
            cnt[r][n-1]   = cnt[r+1][n-1]

        for r in range(n-2, -1, -1):
            for c in range(n-2, -1, -1):
                if board[r][c] == 'X': continue
                val  = 0 if board[r][c] == 'S' else int(board[r][c])
                best, paths = -1, 0
                # Merge three successor directions
                best, paths = merge(score[r][c+1],   cnt[r][c+1],   best, paths)
                best, paths = merge(score[r+1][c],   cnt[r+1][c],   best, paths)
                best, paths = merge(score[r+1][c+1], cnt[r+1][c+1], best, paths)
                if best >= 0:
                    score[r][c] = best + val
                    cnt[r][c]   = paths % MOD

        if cnt[0][0] == 0: return [0, 0]
        return [score[0][0], cnt[0][0] % MOD]

### Go

In [ ]:
func pathsWithMaxScore(board []string) []int {
    const MOD = 1_000_000_007
    n := len(board)
    score := make([][]int, n)
    cnt   := make([][]int64, n)
    for i := range score {
        score[i] = make([]int, n)
        cnt[i]   = make([]int64, n)
    }
    cnt[n-1][n-1] = 1

    merge := func(sNew int, cNew int64, best *int, paths *int64) {
        if cNew == 0 { return }
        if sNew > *best  { *best = sNew; *paths = cNew } else if sNew == *best { *paths += cNew }
    }

    for c := n-2; c >= 0; c-- {
        if board[n-1][c] == 'X' || cnt[n-1][c+1] == 0 { continue }
        val := 0; if board[n-1][c] != 'S' { val = int(board[n-1][c]-'0') }
        score[n-1][c] = score[n-1][c+1] + val; cnt[n-1][c] = cnt[n-1][c+1]
    }
    for r := n-2; r >= 0; r-- {
        if board[r][n-1] == 'X' || cnt[r+1][n-1] == 0 { continue }
        val := 0; if board[r][n-1] != 'S' { val = int(board[r][n-1]-'0') }
        score[r][n-1] = score[r+1][n-1] + val; cnt[r][n-1] = cnt[r+1][n-1]
    }
    for r := n-2; r >= 0; r-- {
        for c := n-2; c >= 0; c-- {
            if board[r][c] == 'X' { continue }
            val := 0; if board[r][c] != 'S' { val = int(board[r][c]-'0') }
            best, paths := -1, int64(0)
            merge(score[r][c+1],   cnt[r][c+1],   &best, &paths)
            merge(score[r+1][c],   cnt[r+1][c],   &best, &paths)
            merge(score[r+1][c+1], cnt[r+1][c+1], &best, &paths)
            if best >= 0 { score[r][c] = best + val; cnt[r][c] = paths % MOD }
        }
    }
    if cnt[0][0] == 0 { return []int{0, 0} }
    return []int{score[0][0], int(cnt[0][0] % MOD)}
}

### Rust

In [ ]:
impl Solution {
    pub fn paths_with_max_score(board: Vec<String>) -> Vec<i32> {
        const MOD: i64 = 1_000_000_007;
        let n = board.len();
        let b: Vec<Vec<u8>> = board.iter().map(|s| s.bytes().collect()).collect();
        let mut score = vec![vec![0i32; n]; n];
        let mut cnt   = vec![vec![0i64; n]; n];
        cnt[n-1][n-1] = 1;

        let mut merge = |s_new: i32, c_new: i64, best: &mut i32, paths: &mut i64| {
            if c_new == 0 { return; }
            if s_new > *best { *best = s_new; *paths = c_new; }
            else if s_new == *best { *paths += c_new; }
        };

        for c in (0..n-1).rev() {
            if b[n-1][c] == b'X' || cnt[n-1][c+1] == 0 { continue; }
            let val = if b[n-1][c] == b'S' { 0 } else { (b[n-1][c] - b'0') as i32 };
            score[n-1][c] = score[n-1][c+1] + val;
            cnt[n-1][c]   = cnt[n-1][c+1];
        }
        for r in (0..n-1).rev() {
            if b[r][n-1] == b'X' || cnt[r+1][n-1] == 0 { continue; }
            let val = if b[r][n-1] == b'S' { 0 } else { (b[r][n-1] - b'0') as i32 };
            score[r][n-1] = score[r+1][n-1] + val;
            cnt[r][n-1]   = cnt[r+1][n-1];
        }
        for r in (0..n-1).rev() {
            for c in (0..n-1).rev() {
                if b[r][c] == b'X' { continue; }
                let val = if b[r][c] == b'S' { 0 } else { (b[r][c] - b'0') as i32 };
                let (mut best, mut paths) = (-1i32, 0i64);
                let (s1,c1) = (score[r][c+1], cnt[r][c+1]);
                let (s2,c2) = (score[r+1][c], cnt[r+1][c]);
                let (s3,c3) = (score[r+1][c+1], cnt[r+1][c+1]);
                merge(s1,c1,&mut best,&mut paths);
                merge(s2,c2,&mut best,&mut paths);
                merge(s3,c3,&mut best,&mut paths);
                if best >= 0 { score[r][c] = best + val; cnt[r][c] = paths % MOD; }
            }
        }
        if cnt[0][0] == 0 { return vec![0,0]; }
        vec![score[0][0], (cnt[0][0] % MOD) as i32]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `board = ["E23","2X2","12S"]`
Paths that avoid the wall in the centre. Best reachable score is 7 with 1 path. Answer: `[7, 1]`.

### 2. Slightly Complex
**Input:** `board = ["E12","1X1","21S"]`
The wall blocks the diagonal/right at (1,1). Only paths circumventing it contribute. Answer: `[4, 2]`.

### 3. Edge Case: Time Factor
**Input:** $100 \times 100$ grid of digits `'9'` with `'S'` and `'E'` corners.
DP fills $100 \times 100 = 10000$ cells, three lookups each — $O(n^2)$ total.

### 4. Edge Case: Space Factor
**Input:** $100 \times 100$ grid.
Two $100 \times 100$ arrays (`score` and `cnt`) — $O(n^2)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `board = ["EX","XS"]`
Every path is blocked by walls. `cnt[0][0]` remains 0; answer is `[0, 0]`.